# Module 3: `optim.py` — Numerically Stable Log-Sum-Exp `CrossEntropy`, `SGD` & `Adam`

This notebook demonstrates overflow-proof `cross_entropy_loss` via the Log-Sum-Exp trick, batch analytical gradient verification, and parameter optimization using `SGD` (with Momentum) and `Adam`.

## 1. Imports & Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import numpy as np
from tensorgrad.tensor import Tensor, numerical_grad_check
from tensorgrad.nn import MLP
from tensorgrad.optim import Adam, SGD, cross_entropy_loss, mse_loss, softmax
print('Loaded tensorgrad.optim successfully.')

Loaded tensorgrad.optim successfully.


## 2. Log-Sum-Exp Numerical Stability on Extreme Logits
Direct evaluation of $e^{1000}$ overflows IEEE-754 `float64`. Subtracting $m_i = \max_j z_{i,j}$ keeps all exponents in $(-\infty, 0]$.

In [2]:
extreme_logits = Tensor([[1000.0, 1001.0, 1000.0]])
loss_ext = cross_entropy_loss(extreme_logits, [1])
loss_ext.backward()
print('Softmax probabilities:', np.round(softmax(extreme_logits)[0], 6))
print(f'Cross-Entropy Loss: {loss_ext.data:.6f}')
print('Analytic gradient dL/dz:', np.round(extreme_logits.grad[0], 6))

Softmax probabilities: [0.211942 0.576117 0.211942]
Cross-Entropy Loss: 0.551445
Analytic gradient dL/dz: [ 0.211942 -0.423883  0.211942]


## 3. Batch Gradient Verification & `Adam` Convergence
We verify the analytical batch gradient $\frac{1}{N}(P - Y)G$ and train an `MLP` with `Adam`.

In [3]:
rng = np.random.default_rng(42)
logits_b = Tensor(rng.normal(size=(6, 4)))
targets_b = np.array([0, 2, 1, 3, 2, 0], dtype=np.int64)
chk = numerical_grad_check(lambda z: cross_entropy_loss(z, targets_b), logits_b)
print(f"Batch Cross-Entropy gradcheck passed: {chk['passed']} | rel_error={chk['rel_error']:.3e}")

X_syn = Tensor(rng.normal(size=(40, 4)))
y_syn = (X_syn.data[:, 0] ** 2 + X_syn.data[:, 1] ** 2 > 1.2).astype(np.int64)
model = MLP(4, [16, 8], 2, rng=rng)
opt = Adam(model.parameters(), lr=0.05)
history = []
for step in range(30):
    opt.zero_grad()
    l = cross_entropy_loss(model(X_syn), y_syn)
    history.append(float(l.data))
    l.backward()
    opt.step()
print('Loss history (first 5 steps):', np.round(history[:5], 4))
print('Loss history (last  5 steps):', np.round(history[-5:], 4))

Batch Cross-Entropy gradcheck passed: True | rel_error=1.637e-10
Loss history (first 5 steps): [0.8168 0.7272 0.5277 0.425  0.4025]
Loss history (last  5 steps): [0.0135 0.0114 0.0092 0.0074 0.0062]


## Conclusion
`optim.py` prevents exponential overflow via Log-Sum-Exp and achieves fast convergence on non-linear decision boundaries with bias-corrected `Adam`.